# Lección 3.1 — El modelo como "función con perillas"

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 70–90 min (lección ≈ 50 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 1.2 (parámetro vs. hiperparámetro). Nada de matemáticas: si sabes multiplicar y sumar, alcanza.

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es un modelo usando la analogía de una **máquina con perillas**.
- Leer los dos **parámetros** de una recta (pendiente e intercepto) en lenguaje de negocio.
- Ajustar perillas **a mano** y entender por qué eso no escala cuando hay miles o millones de perillas.
- Distinguir con seguridad **parámetros** (los aprende el modelo) de **hiperparámetros** (los eliges tú).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: una máquina con perillas

Imagina una **radio antigua**. Tiene perillas: volumen, sintonía, graves, agudos. La radio siempre funciona igual por dentro; lo único que cambia el sonido es **dónde dejas cada perilla**.

```
            ┌──────────────────────────────┐
 entrada ──►│  🎛️  🎛️   MODELO   🎛️  🎛️   │──► salida (predicción)
 (temperatura)└──────────────────────────────┘     (helados vendidos)
                  ▲ perillas = PARÁMETROS
```

**Un modelo de Machine Learning es exactamente eso:** una fórmula fija con perillas. Le das una entrada (la temperatura de hoy), hace una cuenta y devuelve una salida (cuántos helados venderás).

- Las **perillas** se llaman **parámetros** (o *pesos*, *weights*).
- **Entrenar** un modelo = **buscar la posición de las perillas** que hace que sus predicciones se parezcan a los datos reales.

Eso es todo. El resto del módulo trata de **cómo** encontrar esa posición sin hacerlo a mano.

### ✋ Ejercicio 1 — Calentamiento ☕
Estás aprendiendo a preparar café en una cafetera nueva. Tiene dos perillas: **gramos de café** y **temperatura del agua**. Preparas una taza, la pruebas, está demasiado amarga, mueves las perillas y repites.

Relaciona cada parte de esta historia con su nombre en ML:

| Historia del café | ¿Qué es en ML? |
|---|---|
| a) Los gramos de café y la temperatura | ? |
| b) La cafetera (lo que hace con las perillas) | ? |
| c) "Está demasiado amarga" | ? |
| d) Preparar, probar y mover las perillas una y otra vez | ? |

Opciones: *modelo · parámetros · error (pérdida) · entrenamiento*

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

a) **Parámetros** (las perillas) · b) **Modelo** (la fórmula fija) · c) **Error** o **pérdida**: cuánto te alejas del resultado que querías (Lección 3.2) · d) **Entrenamiento**: el ciclo de probar, medir el error y corregir (Lecciones 3.3–3.5).

Fíjate en que nunca cambias la cafetera: solo las perillas. Lo mismo hace un modelo.
</details>

---
## 2. 📖 El concepto: la recta, el modelo más simple

Una heladería anotó durante 30 días la **temperatura máxima** y los **helados vendidos**. Mira los datos:

In [ ]:
# ▶️ Datos de la heladería (30 días)
rng = np.random.default_rng(0)
temperatura = rng.uniform(14, 36, 30).round(1)                          # °C
ventas = (8 * temperatura - 60 + rng.normal(0, 15, 30)).round()        # helados vendidos

plt.scatter(temperatura, ventas, color="tab:blue", edgecolor="k")
plt.xlabel("Temperatura máxima (°C)"); plt.ylabel("Helados vendidos")
plt.title("La heladería: 30 días de datos"); plt.grid(alpha=.3); plt.show()

Se ve una tendencia: **más calor → más helados**, y los puntos forman casi una línea. El modelo más simple que existe para esto es una **recta**:

$$\text{ventas} = w \times \text{temperatura} + b$$

Tiene **dos perillas**:

| Perilla | Nombre técnico | Qué significa en el negocio |
|---|---|---|
| `w` | **peso** o **pendiente** (*weight*) | Cuántos helados **más** vendes por cada grado extra |
| `b` | **sesgo** o **intercepto** (*bias*) | El punto de partida: lo que diría el modelo a 0 °C |

Con `w = 8` y `b = -60` el modelo dice: *"multiplica la temperatura por 8 y réstale 60"*.

### 🔮 Ejercicio 2 — Usa el modelo a mano
Con las perillas en `w = 8` y `b = -60`, ¿cuántos helados predice el modelo para un día de **30 °C**? ¿Y para uno de **20 °C**? Hazlo en papel y luego ejecuta.

✍️ *Mis predicciones:* 30 °C → ___ · 20 °C → ___

In [ ]:
def modelo(temp, w, b):
    # La "máquina": una cuenta fija. Lo único que cambia son las perillas w y b.
    return w * temp + b

print("30 °C →", modelo(30, w=8, b=-60), "helados")
print("20 °C →", modelo(20, w=8, b=-60), "helados")

<details><summary><b>👀 Ver respuesta</b></summary>

- 30 °C → 8 × 30 − 60 = **180** helados.
- 20 °C → 8 × 20 − 60 = **100** helados.

Diez grados más dan 80 helados más: eso es justo lo que significa `w = 8` (8 helados por grado).
</details>

---
## 3. 📖 Cómo se "entrena": mover perillas hasta que encaje

Vamos a hacer de entrenadores. La función `probar_perillas` dibuja la recta que producen tus perillas sobre los datos y te dice **en cuántos helados te equivocas, en promedio**. Ese número es tu brújula: cuanto más bajo, mejor.

> 📝 En la Lección 3.2 le daremos nombre formal a ese número (la **función de pérdida**). Por ahora basta con saber que mide "qué tan lejos" quedan tus predicciones de la realidad.

In [ ]:
def probar_perillas(w, b):
    pred = modelo(temperatura, w, b)
    error_medio = np.mean(np.abs(ventas - pred))       # promedio de "cuánto me equivoqué"
    t = np.linspace(10, 40, 100)
    plt.scatter(temperatura, ventas, edgecolor="k", label="Datos reales")
    plt.plot(t, modelo(t, w, b), "r-", lw=2, label=f"Modelo: {w}·temp + {b}")
    plt.vlines(temperatura, ventas, pred, color="gray", alpha=.5, label="Errores")
    plt.xlabel("Temperatura (°C)"); plt.ylabel("Helados"); plt.ylim(-50, 320)
    plt.title(f"Error medio: {error_medio:.1f} helados por día"); plt.legend(); plt.grid(alpha=.3); plt.show()

probar_perillas(w=0, b=100)

Con `w = 0` la recta es **horizontal**: el modelo dice "100 helados" haga el calor que haga. Las líneas grises son los errores de cada día.

### ✏️ Ejercicio 3 — Sé tú el algoritmo de entrenamiento
Cambia `w` y `b` en la celda de abajo y vuelve a ejecutarla varias veces. **Meta: dejar el error medio por debajo de 15 helados.**

Estrategia sugerida: primero gira `w` hasta que la recta tenga la **inclinación** de la nube de puntos; después mueve `b` para **subirla o bajarla** entera.

In [ ]:
probar_perillas(w=0, b=100)      # ✏️ cambia estos dos números y vuelve a ejecutar

✍️ *Mis mejores perillas:* w = ___ · b = ___ · error = ___

<details><summary><b>👀 Ver respuesta</b></summary>

Algo como **`w = 8`, `b = -60`** deja el error en unos **12 helados**. La recta que encontraremos sin adivinar en la Lección 3.3 tiene `w ≈ 8.0`, `b ≈ -56` con un error de **≈ 11.5**: tu búsqueda a mano probablemente quedó muy cerca.

Fíjate en **cómo** buscaste: moviste una perilla, miraste si el error bajaba y decidiste hacia dónde seguir. **Ese es exactamente el algoritmo de descenso de gradiente** (Lección 3.3), solo que la computadora lo hace miles de veces por segundo y sin mirar la gráfica.
</details>

### 🔮 Ejercicio 4 — ¿Qué hace cada perilla?
Sin ejecutar, describe cómo se verá la recta en cada caso:
1. `w = 0, b = 150`
2. `w = 8, b = 0`  (comparada con `w = 8, b = -60`)
3. `w = -5, b = 250`

Luego compruébalo con `probar_perillas`.

✍️ *Tu respuesta:*

In [ ]:
probar_perillas(w=-5, b=250)     # ✏️ prueba los tres casos

<details><summary><b>👀 Ver respuesta</b></summary>

1. Recta **horizontal** a la altura 150: el modelo ignora la temperatura.
2. Misma inclinación, pero **60 helados más arriba**: `b` sube o baja la recta sin girarla.
3. Recta **descendente**: el modelo diría que con más calor se venden *menos* helados. Contradice los datos, así que el error es enorme.

Resumen: **`w` gira la recta, `b` la desplaza.**
</details>

---
## 4. 📖 ¿Qué pasa si hay muchas perillas?

Con 2 perillas, mover a mano funciona. Pero los modelos reales tienen **muchas más**:

| Modelo | Número aproximado de perillas |
|---|---|
| Recta con 1 característica | 2 |
| Recta con 10 características (Lección 5.1) | 11 (un `w` por característica + `b`) |
| Red neuronal pequeña para dígitos (Módulo 9) | ~100,000 |
| GPT-3 (Módulo 13) | 175,000,000,000 |

Una idea tentadora: **probar combinaciones al azar** y quedarse con la mejor. Probemos con nuestras 2 perillas.

In [ ]:
rng_busqueda = np.random.default_rng(1)
N_INTENTOS = 1000                                   # ✏️ prueba 10, 100, 10000

mejor = (None, None, np.inf)
for _ in range(N_INTENTOS):
    w = rng_busqueda.uniform(-20, 20)
    b = rng_busqueda.uniform(-200, 200)
    error = np.mean(np.abs(ventas - modelo(temperatura, w, b)))
    if error < mejor[2]:
        mejor = (w, b, error)

print(f"Mejor de {N_INTENTOS} intentos al azar: w = {mejor[0]:.2f}, b = {mejor[1]:.1f}, error = {mejor[2]:.1f}")

Con 2 perillas y 1,000 intentos se llega cerca. Pero ¿y con más perillas?

### 🔮 Ejercicio 5 — La explosión de combinaciones
Supón que, para cada perilla, pruebas solo **10 posiciones**. Con 2 perillas son 10 × 10 = 100 combinaciones.
1. ¿Cuántas combinaciones hay con **10 perillas**?
2. Si tu computadora prueba **un millón** de combinaciones por segundo, ¿cuánto tarda?
3. ¿Y con 100 perillas?

✍️ *Tu respuesta:*

In [ ]:
PRUEBAS_POR_SEGUNDO = 1_000_000
for perillas in [2, 5, 10, 20, 100]:
    combinaciones = 10 ** perillas
    segundos = combinaciones / PRUEBAS_POR_SEGUNDO
    print(f"{perillas:>3} perillas → {combinaciones:.0e} combinaciones → {segundos / 3600:.1e} horas")

<details><summary><b>👀 Ver respuesta</b></summary>

1. 10¹⁰ = **diez mil millones** de combinaciones.
2. 10,000 segundos ≈ **2.8 horas**. Y eso con solo 10 posiciones por perilla, que es una búsqueda muy burda.
3. 10¹⁰⁰ combinaciones: muchísimo más que la edad del universo. **Imposible.**

Por eso nadie entrena modelos probando al azar. Hace falta una forma **inteligente** de saber hacia dónde mover cada perilla: eso es el **descenso de gradiente** (Lección 3.3), y funciona igual con 2 perillas que con 175 mil millones.
</details>

---
## 5. 📖 ¿Qué pasa si el modelo tiene la forma equivocada?

Las perillas solo ajustan la **posición** de la fórmula, no su **forma**. Una recta, por mucho que gires sus perillas, siempre será una recta.

Una cafetería anotó cuántos **cafés helados** vende según la temperatura. Con mucho calor la gente deja de salir a la calle, y las ventas bajan. Ejecuta: la celda busca **las mejores perillas posibles** para una recta (2 perillas) y para una parábola (3 perillas).

In [ ]:
rng_cafe = np.random.default_rng(2)
temp_cafe = rng_cafe.uniform(10, 40, 40)
cafes = -0.5 * (temp_cafe - 28) ** 2 + 120 + rng_cafe.normal(0, 8, 40)

recta = np.polyfit(temp_cafe, cafes, deg=1)      # mejores 2 perillas para una recta
parabola = np.polyfit(temp_cafe, cafes, deg=2)   # mejores 3 perillas para una parábola

t = np.linspace(10, 40, 100)
plt.scatter(temp_cafe, cafes, edgecolor="k", label="Datos")
for perillas, nombre, color in [(recta, "Recta (2 perillas)", "red"), (parabola, "Parábola (3 perillas)", "green")]:
    error = np.mean(np.abs(cafes - np.polyval(perillas, temp_cafe)))
    plt.plot(t, np.polyval(perillas, t), color=color, lw=2, label=f"{nombre}: error medio {error:.1f}")
plt.xlabel("Temperatura (°C)"); plt.ylabel("Cafés helados"); plt.legend(); plt.grid(alpha=.3); plt.show()

### ✋ Ejercicio 6
1. ¿Por qué la recta no puede bajar su error aunque tenga las *mejores* perillas posibles?
2. Elegir "recta" o "parábola", ¿es un **parámetro** o un **hiperparámetro**?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Porque el problema no está en **dónde** están las perillas, sino en la **forma** del modelo: una recta no puede subir y luego bajar. Esto se llama **subajuste** (*underfitting*, Lección 4.4): el modelo es demasiado simple para el patrón.
2. Un **hiperparámetro**. Lo decides **tú** antes de entrenar. El entrenamiento solo mueve las perillas (parámetros) dentro de la forma que elegiste.
</details>

---
## 6. 📖 Parámetros vs. hiperparámetros: la diferencia clave

| | **Parámetros** | **Hiperparámetros** |
|---|---|---|
| ¿Quién los elige? | El **algoritmo de entrenamiento**, a partir de los datos | **Tú**, antes de entrenar |
| Analogía | Dónde quedan las perillas de la radio | Qué radio compras y cuánto tiempo pasas ajustándola |
| Ejemplos | `w` y `b` de la recta; las preguntas de un árbol (Lección 5.4); los pesos de una red | Recta o parábola; `max_depth` de un árbol; **tasa de aprendizaje**, **épocas** y **tamaño de lote** (Lección 3.4) |
| ¿Se guardan en el modelo final? | Sí: **son** el modelo | No: son la "receta" para obtenerlo |

### ✋ Ejercicio 7 — Clasifica
¿Parámetro (P) o hiperparámetro (H)?
1. El número de vecinos `k` en KNN.
2. El valor `w = 7.98` que encontró el entrenamiento.
3. Cuántas veces el algoritmo recorre los datos durante el entrenamiento.
4. El umbral `deuda_pct <= 40.5` que eligió un árbol.
5. El grado del polinomio (recta = 1, parábola = 2).

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuesta</b></summary>

1. **H** — lo eliges tú.  2. **P** — lo aprendió de los datos.  3. **H** — se llama número de **épocas** (Lección 3.4).  4. **P** — el árbol lo descubrió solo.  5. **H** — es la forma del modelo.

Pregunta rápida para decidir: *"¿Este número salió de los datos o lo escribí yo?"*
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Confundir parámetros con hiperparámetros | Intentas "ajustar a mano" lo que el modelo aprende solo, o esperas que aprenda solo lo que debes elegir tú | Usa la pregunta: *¿salió de los datos o lo escribí yo?* |
| Creer que más perillas siempre es mejor | Con demasiadas perillas el modelo **memoriza** el ruido (sobreajuste, Lección 4.4) | Empieza simple y añade complejidad solo si el error en datos nuevos baja |
| Interpretar `w` sin unidades | "w = 8" no dice nada sin "helados **por grado**" | Siempre lee un parámetro con sus unidades |
| Usar el modelo fuera del rango de los datos | La recta se aprendió entre 14 °C y 36 °C; fuera de ahí puede decir disparates | Desconfía de predicciones lejos de lo que el modelo vio |

### 🔮 Ejercicio 8 — Un día de invierno
Con `w = 8` y `b = -60`, ¿cuántos helados predice el modelo para un día de **5 °C**? ¿Tiene sentido?

✍️ *Tu respuesta:*

In [ ]:
print("5 °C →", modelo(5, w=8, b=-60), "helados")

<details><summary><b>👀 Ver respuesta</b></summary>

8 × 5 − 60 = **−20 helados**. ¡Imposible! El modelo nunca vio días por debajo de 14 °C y simplemente prolonga la recta. Es el último error de la tabla: **un modelo solo es confiable en la zona donde tuvo datos.**
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. Un modelo es una **fórmula fija con perillas**: entra un dato, sale una predicción.
2. Las perillas se llaman **parámetros** (pesos). En una recta son `w` (gira) y `b` (desplaza).
3. **Entrenar = encontrar la posición de las perillas** que hace que las predicciones se parezcan a los datos.
4. Buscar a mano o al azar funciona con 2 perillas, pero **explota** con muchas: hace falta un método inteligente (Lección 3.3).
5. Los **hiperparámetros** los eliges tú (la forma del modelo, cuánto entrenar…); los **parámetros** los aprende el modelo.

---
## 9. 🏋️ Ejercicios finales

Estos ejercicios son más largos (≈30 min) y combinan todo lo anterior.

### 9.1 Repaso rápido (opción múltiple)

**1.** En el modelo `ventas = w · temperatura + b`, si subes `b` en 20 y dejas `w` igual, la recta…
- a) se inclina más · b) sube 20 unidades sin girar · c) baja 20 unidades · d) se vuelve horizontal

**2.** Un modelo de precios de casas usa 6 características (m², recámaras, baños, antigüedad, piso, distancia al metro) en una fórmula lineal. ¿Cuántos parámetros tiene?
- a) 6 · b) 7 · c) 12 · d) 2

**3.** ¿Cuál es un **hiperparámetro**?
- a) El peso de la característica "m²" · b) El intercepto `b` · c) El número de épocas · d) El umbral que aprendió un árbol

**4.** ¿Por qué no se entrenan los modelos probando combinaciones al azar?
- a) Porque es ilegal · b) Porque el número de combinaciones explota con muchas perillas · c) Porque el azar no existe en las computadoras · d) Porque siempre da el mismo resultado

**5.** Tu recta tiene las **mejores** perillas posibles y aun así se equivoca mucho, siempre de la misma forma (subestima en los extremos y sobreestima en el centro). ¿Qué cambiarías?
- a) Volver a entrenar con otras perillas iniciales · b) La forma del modelo (un hiperparámetro) · c) Nada, es ruido · d) Quitar datos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** `b` desplaza la recta; `w` la gira.
2. **b)** Un `w` por característica (6) más el intercepto `b`: **7**.
3. **c)** Las épocas las eliges tú (Lección 3.4). Las demás salen de los datos.
4. **b)** Con 10 posiciones por perilla y 100 perillas serían 10¹⁰⁰ combinaciones.
5. **b)** Si ya tiene las mejores perillas, el problema es la forma: un error **sistemático** delata **subajuste**. Una forma más flexible (p. ej. una parábola) puede capturarlo.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un compañero dice: *"Un modelo con 175 mil millones de perillas siempre será mejor que uno con 2"*. Da un argumento a favor y uno en contra.

**B. Caso de negocio 💼** La heladería encontró `w = 8` y `b = −56`. El gerente pregunta: *"¿Cuántos helados de más debo tener listos por cada grado extra que anuncie el pronóstico?"* y *"¿Puedo usar el modelo para planear diciembre, cuando hace 8 °C?"*. ¿Qué le respondes?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** A favor: más perillas permiten capturar patrones mucho más complejos (lenguaje, imágenes) que una recta jamás podría. En contra: más perillas necesitan **muchos más datos** y cómputo, y con pocos datos tienden a **memorizar** (sobreajuste). Para predecir helados con la temperatura, 2 perillas bastan.

**B.** 1) **Unos 8 helados más por cada grado**: es justo lo que significa `w`. 2) **No conviene**: los datos van de 14 °C a 36 °C; a 8 °C el modelo diría 8 × 8 − 56 = 8 helados, pero nunca vio un día así y la relación podría ser distinta (quizá la gente compra helado igual en las fiestas). Lo honesto es decir *"el modelo no sabe"* y juntar datos de invierno.
</details>

### 9.3 🧩 Práctica integradora: la tarifa del taxi 🚕

Una app de taxis te pasa 50 viajes con su distancia (km) y lo que costaron ($). Quieren saber **cuánto cobran de banderazo** (lo que pagas solo por subirte) y **cuánto por kilómetro**. Nadie se los dijo: tendrás que descubrirlo **ajustando perillas**.

Lo harás en 5 pasos cortos. Revisa cada respuesta antes de avanzar.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng_taxi = np.random.default_rng(5)
km = rng_taxi.uniform(1, 20, 50).round(1)
tarifa = (35 + 9 * km + rng_taxi.normal(0, 6, 50)).round()

plt.scatter(km, tarifa, edgecolor="k"); plt.xlabel("Distancia (km)"); plt.ylabel("Tarifa ($)")
plt.title("50 viajes en taxi"); plt.grid(alpha=.3); plt.show()

**Paso 2 🔮 — Lee la gráfica.** El modelo será `tarifa = w · km + b`. Mirando la nube de puntos, estima a ojo:
- ¿Cuánto vale la tarifa cuando la distancia es casi 0? Eso es aproximadamente `b`.
- ¿Cuánto sube la tarifa entre 5 km y 15 km? Divide entre 10 y tendrás `w`.

✍️ *Mis estimaciones:* b ≈ ___ · w ≈ ___

<details><summary><b>👀 Ver respuesta</b></summary>

A ojo: cerca de 0 km la tarifa ronda **$35–45** (así que `b` ≈ 40) y entre 5 km y 15 km sube de ≈ $80 a ≈ $170, unos $90 en 10 km, así que `w` ≈ **9**. Una estimación visual nunca es exacta; para eso están los siguientes pasos.
</details>

**Paso 3 ✏️ — Ajusta las perillas.** Usa tus estimaciones como punto de partida y muévelas hasta que el error medio baje de **$6**.

In [ ]:
def probar_taxi(w, b):
    error = np.mean(np.abs(tarifa - (w * km + b)))
    d = np.linspace(0, 21, 50)
    plt.scatter(km, tarifa, edgecolor="k"); plt.plot(d, w * d + b, "r-", lw=2)
    plt.title(f"tarifa = {w}·km + {b}   →   error medio ${error:.2f}"); plt.grid(alpha=.3); plt.show()

probar_taxi(w=9, b=40)        # ✏️

<details><summary><b>👀 Ver respuesta</b></summary>

Con `w = 9, b = 40` el error medio es de **≈ $6.6**: la recta queda un poco alta. Bajando `b` a 36 llega a **≈ $4.8**. Cualquier combinación cerca de `w ≈ 9` y `b ≈ 33–36` es muy buena.
</details>

**Paso 4 ▶️ — Compara con el mejor ajuste posible.** `np.polyfit` calcula directamente las mejores perillas para una recta (en la Lección 3.3 verás que el descenso de gradiente llega al mismo lugar paso a paso).

In [ ]:
w_mejor, b_mejor = np.polyfit(km, tarifa, deg=1)
error_mejor = np.mean(np.abs(tarifa - (w_mejor * km + b_mejor)))
print(f"Mejores perillas: w = {w_mejor:.2f} $/km · b = ${b_mejor:.2f} · error medio = ${error_mejor:.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

El mejor ajuste da **w ≈ 9.0 $/km** y **b ≈ $33.8**, con un error medio de **≈ $4.35**. Los valores "verdaderos" con los que se generaron los datos eran 9 y 35: el modelo los **descubrió a partir de los ejemplos**, sin que nadie se los dijera.
</details>

**Paso 5 ✋ — Tradúcelo al negocio.**
1. Explica `w` y `b` a alguien que no sabe ML, en una frase cada uno.
2. ¿Cuánto costaría un viaje de **12 km**?
3. ¿Confiarías en el modelo para un viaje de **80 km** al aeropuerto de otra ciudad? ¿Por qué?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. `b ≈ $34`: *"solo por subirte pagas unos 34 pesos (el banderazo)"*. `w ≈ $9`: *"después, cada kilómetro suma unos 9 pesos"*.
2. 9.04 × 12 + 33.78 ≈ **$142**.
3. **No del todo**: los datos solo llegan a 20 km. En viajes largos podría haber tarifas distintas (peajes, tarifa nocturna, tarifa plana al aeropuerto). El modelo solo es confiable dentro del rango que vio (Ejercicio 8).
</details>

---
## 📚 Para profundizar
- 📺 **Video:** 3Blue1Brown, *"But what is a neural network?"* (inglés, con subtítulos): los "pesos" como perillas de una red.
- 🎮 **Interactivo:** *TensorFlow Playground* (playground.tensorflow.org): mira cómo cambian los pesos mientras la red aprende.
- 📘 **Lectura:** Google, *Machine Learning Crash Course*, sección *Linear regression*.

---
⬅️ **Anterior:** [2.9 Proyecto del módulo: Los datos sucios de CineNube 🎬](../../nivel-1-cero/m02-datos/2.9-proyecto-del-modulo.md) · ➡️ **Siguiente:** [3.2 Función de pérdida: medir qué tan equivocado está el modelo](3.2-funcion-de-perdida.ipynb)